# 第11课 - 模型上下文协议 (MCP)

**模型上下文协议 (MCP)** 是一个开放标准，使代理能够在运行时动态发现并使用工具、资源和数据源。MCP 允许代理连接到按需公开功能的外部服务器，而不是将工具硬编码到代理中。

在本课中，你将学习：
- MCP 是什么以及它为何对代理系统很重要
- MCP 客户端-服务器架构如何工作
- 如何构建使用 MCP 风格进行工具发现的代理


## 设置

**先决条件：**
- 拥有已部署模型的 Azure AI Foundry 项目
- 运行 `az login` 进行身份验证


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什么是模型上下文协议 (MCP)?

MCP 定义了一种标准方式，供 AI 代理发现并与外部工具和数据源交互：

- **MCP 服务器**: 通过标准协议公开工具、资源和提示
- **MCP 客户端**: 连接到服务器并发现可用功能的代理运行时
- **动态发现**: 代理不需要硬编码的工具 — 它们在运行时发现可用能力

这对于构建可扩展的代理系统非常有用，在这样的系统中可以添加新功能而无需修改代理代码。


## MCP 的工作原理

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. 代理（MCP 客户端）连接到 MCP 服务器
2. 服务器返回一份包含可用工具及其模式的列表
3. 随后代理可以在推理过程中调用任何被发现的工具
4. 结果通过相同的协议返回


## 模拟 MCP 工具发现

由于真实的 MCP 服务器需要一个运行中的服务器进程，我们将使用 `@tool` 函数来演示该模式，这些函数模拟了与 MCP 连接的住宿服务所提供的功能。

在生产环境中，这些工具会从 MCP 服务器动态发现，而不是在本地定义。


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## 使用 MCP 风格工具构建代理


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP 在生产环境中

在生产环境中，MCP 支持强大的模式：

- **动态工具发现**: 代理连接到 MCP 服务器并在运行时发现工具
- **解耦架构**: 工具提供方可以独立于代理进行更新
- **跨组织共享**: 团队可以通过 MCP 服务器公开功能，任何代理都可以使用
- **Microsoft Agent Framework 支持**: MAF 通过 `mcp` 集成内置对 MCP 客户端的支持

要在 MAF 中使用真实的 MCP 服务器，您可以通过 `hosted_mcp_tool()` 或 MCP 客户端集成进行连接。

**了解更多：**
- [MCP 规范](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP 支持](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## 总结

在本课中，您学到了：
- **MCP** 是一个开放标准，用于在代理和工具提供者之间进行动态工具发现
- **客户端-服务器架构** 让代理在运行时发现能力
- MCP 使 **可扩展、解耦的代理系统** 成为可能，其中可以在不更改代码的情况下添加工具
- Microsoft Agent Framework 提供 **内置的 MCP 支持**，适用于生产环境


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免责声明：

本文件由 AI 翻译服务 Co-op Translator (https://github.com/Azure/co-op-translator) 翻译。尽管我们尽力保证准确性，但请注意自动翻译可能包含错误或不准确之处。原始语言的原文应被视为权威来源。对于重要信息，建议使用专业人工翻译。本公司/我们不对因使用本翻译而导致的任何误解或错误解释承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
